In [ ]:
# Import required libraries for RMD-TVReg

import numpy as np
import pandas as pd
from scipy.stats import multivariate_normal
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
import matplotlib.pyplot as plt

np.random.seed(123)


# Section 4.1 Simulation Setup
# This block implements the data-generating mechanism described
# Time-varying regression: y_t = x_t^T beta_t + epsilon_t
# Dynamic coefficients: smooth sine/cosine patterns
# Heavy-tailed noise + contamination
# Missing data (MCAR) in both X and y


import numpy as np

def generate_time_varying_data(T=200,
                               p=50,
                               noise_scale=1.0,
                               noise_df=4,
                               contamination=0.1,
                               missing_rate=0.1,
                               scenario='smooth'):
    """
    Generate synthetic dynamic regression data with:
    - Smooth or abrupt time-varying coefficients
    - t-distribution noise + contamination
    - Missing values in X and y (MCAR)

    Args:
        scenario: 'smooth' or 'abrupt'
    """

    # Predictor matrix
    X = np.random.normal(0, 1, size=(T, p))

    # Time index
    t = np.linspace(0, 1, T)

    # True coefficients
    beta = np.zeros((T, p))

    # ===============================
    # Scenario A: Smooth dynamics
    # ===============================
    if scenario == 'smooth':
        beta[:, 0] = 1.2 * np.sin(4 * np.pi * t)
        beta[:, 1] = 0.8 * np.cos(2 * np.pi * t)

    # ===============================
    # Scenario B: Abrupt change
    # ===============================
    elif scenario == 'abrupt':
        # Sharp structural break at midpoint
        beta[:T//2, 0] = 1.0
        beta[T//2:, 0] = -1.0

        # second coefficient is constant (simpler baseline)
        beta[:, 1] = 0.5

    else:
        raise ValueError("scenario must be 'smooth' or 'abrupt'")

    
    # Heavy-tailed noise
    
    eps = noise_scale * np.random.standard_t(df=noise_df, size=T)

   
    # Contamination (outliers)
    
    contam_idx = np.random.choice(T, int(contamination * T), replace=False)
    eps[contam_idx] += np.random.normal(8, 2, size=len(contam_idx))

    
    # Response
    
    y = np.sum(X * beta, axis=1) + eps

   
    # Missingness (MCAR)
    
    mask = np.random.rand(T, p) < missing_rate
    X[mask] = np.nan

    y[np.random.rand(T) < missing_rate] = np.nan

    return X, y, beta
     

#  RMD-TVReg EM + Coordinate Descent Estimation:
# ==============================================================
# This block implements:
#   E-step: simple conditional mean imputation for missing X, y
#   M-step: robust regression updates with SCAD/MCP-style weights
#   Temporal smoothing applied via fused-L2 penalty
#   Coordinate descent across time and features

import numpy as np

def huber_loss_derivative(r, tau):
    """Derivative of Huber loss."""
    return np.where(np.abs(r) <= tau, r, tau * np.sign(r))

def scad_weight(beta, lam, a=3.7):
    """Local linear weight for SCAD penalty."""
    absb = np.abs(beta)
    w = np.zeros_like(beta)
    for i in range(len(beta)):
        if absb[i] <= lam:
            w[i] = lam
        elif absb[i] <= a * lam:
            w[i] = (a * lam - absb[i]) / (a - 1)
        else:
            w[i] = 0
    return w

def rmd_tvreg_fit(X, y, lam1=0.1, lam2=0.1, tau=1.0,
                  max_iter=20, cd_iter=5, is_static=False):
    """
    Fit the RMD-TVReg model using EM + coordinate descent.

    Args:
        X: shape (T, p)
        y: shape (T,)
        lam1: sparsity penalty
        lam2: temporal smoothness penalty
        tau: Huber tuning parameter
        max_iter: number of outer EM iterations
        cd_iter: number of inner coordinate descent iterations
        is_static: if True, force coefficients to be constant across time
                   (used for Static-Huber baseline)

    Returns:
        beta_est: estimated coefficients, shape (T, p)
    """

    T, p = X.shape

    # Initialize coefficients
    beta = np.zeros((T, p))

    for k in range(max_iter):

        # E-step: simple mean imputation
        X_imp = np.where(np.isnan(X), np.nanmean(X, axis=0), X)
        y_imp = np.where(np.isnan(y), np.nanmean(y), y)

        # M-step: coordinate descent
        for cd in range(cd_iter):
            for t in range(T):
                # residual
                r = y_imp[t] - X_imp[t] @ beta[t]
                grad = -huber_loss_derivative(r, tau) * X_imp[t]

                # SCAD weights for sparsity
                w_scad = scad_weight(beta[t], lam1)

                # gradient step
                beta[t] = beta[t] - 0.01 * (grad + w_scad * np.sign(beta[t]))

                # temporal smoothing only for dynamic model
                if not is_static:
                    if t > 0:
                        beta[t] -= lam2 * (beta[t] - beta[t - 1])
                    if t < T - 1:
                        beta[t] -= lam2 * (beta[t] - beta[t + 1])

            # If static baseline, force all beta_t to share one common vector
            if is_static:
                beta_mean = np.mean(beta, axis=0)
                beta[:] = beta_mean

    return beta

def calculate_tv(beta_path):
    diffs = np.diff(beta_path, axis=0)
    return np.sum(np.linalg.norm(diffs, axis=1))     

# Single Simulation Run for RMD-TVReg:
# ==============================================================
# This block:
#Generates one synthetic dataset with time-varying coefficients
#Adds missing values and contamination
#Fits the RMD-TVReg estimator
#Computes coefficient estimation error (MSE)
#Plots the true vs. estimated trajectory for one coefficient
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt


# Simulation settings

T = 200
p = 50
missing_rate = 0.1
contamination = 0.1

# Choose scenario: 'smooth' or 'abrupt'
scenario = 'smooth'

# Generate synthetic data
X_sim, y_sim, beta_true = generate_time_varying_data(
    T=T,
    p=p,
    noise_scale=1.0,
    noise_df=4,
    contamination=contamination,
    missing_rate=missing_rate,
    scenario=scenario
)

# Fit RMD-TVReg
beta_est = rmd_tvreg_fit(
    X=X_sim,
    y=y_sim,
    lam1=0.1,
    lam2=0.3,
    tau=2.0,
    max_iter=40,
    cd_iter=15,
    is_static=False
)

# Compute MSE
mask = ~np.isnan(beta_est).any(axis=1)

if np.sum(mask) == 0:
    print("All rows contain NaN — estimation diverged. Try smaller lam2 or fewer iterations.")
else:
    mse_beta = mean_squared_error(
        beta_true[mask].flatten(),
        beta_est[mask].flatten()
    )
    tv_beta = calculate_tv(beta_est)

    print(f"Scenario: {scenario}")
    print(f"Coefficient MSE (RMD-TVReg): {mse_beta:.4f}")
    print(f"Total Variation (RMD-TVReg): {tv_beta:.4f}")

# Plot trajectory for beta_1(t)
plt.figure(figsize=(9, 5))
plt.plot(beta_true[:, 0], label="True β1", linewidth=3)
plt.plot(beta_est[:, 0], label="Estimated β1 (RMD-TVReg)", linewidth=3)

plt.xlabel("Time", fontsize=18)
plt.ylabel("Coefficient value", fontsize=18)
plt.title(f"Time-varying coefficient trajectory: β1 ({scenario})", fontsize=20)

plt.xticks(fontsize=16)
plt.yticks(fontsize=16)

plt.legend(fontsize=16)
plt.tight_layout()
plt.show()
     

import numpy as np
from sklearn.metrics import mean_squared_error


# Helper: run one simulation for given contamination/missing
def run_one_simulation(T=200,
                       p=50,
                       missing_rate=0.1,
                       contamination=0.1,
                       lam1=0.1,
                       lam2=0.3,
                       tau_robust=2.0,
                       tau_ls=1e6,
                       scenario='smooth'):
    """
    Run one simulation replication.

    Returns:
        - MSE and TV for Static-Huber
        - MSE and TV for TV-LS
        - MSE and TV for RMD-TVReg
    """

    # Generate synthetic data
    X_sim, y_sim, beta_true = generate_time_varying_data(
        T=T,
        p=p,
        noise_scale=1.0,
        noise_df=4,
        contamination=contamination,
        missing_rate=missing_rate,
        scenario=scenario
    )

    
    # Method 1: Static-Huber baseline
    
    beta_est_static = rmd_tvreg_fit(
        X=X_sim,
        y=y_sim,
        lam1=lam1,
        lam2=0.0,
        tau=tau_robust,
        max_iter=40,
        cd_iter=15,
        is_static=True
    )

    mask_static = ~np.isnan(beta_est_static).any(axis=1)
    coef_mse_static = mean_squared_error(
        beta_true[mask_static].flatten(),
        beta_est_static[mask_static].flatten()
    )
    tv_static = calculate_tv(beta_est_static)

    
    # Method 2: TV-LS baseline (dynamic but non-robust)
    
    beta_est_ls = rmd_tvreg_fit(
        X=X_sim,
        y=y_sim,
        lam1=lam1,
        lam2=lam2,
        tau=tau_ls,
        max_iter=40,
        cd_iter=15,
        is_static=False
    )

    mask_ls = ~np.isnan(beta_est_ls).any(axis=1)
    coef_mse_ls = mean_squared_error(
        beta_true[mask_ls].flatten(),
        beta_est_ls[mask_ls].flatten()
    )
    tv_ls = calculate_tv(beta_est_ls)

    
    # Method 3: RMD-TVReg (dynamic + robust)
    
    beta_est_robust = rmd_tvreg_fit(
        X=X_sim,
        y=y_sim,
        lam1=lam1,
        lam2=lam2,
        tau=tau_robust,
        max_iter=40,
        cd_iter=15,
        is_static=False
    )

    mask_rb = ~np.isnan(beta_est_robust).any(axis=1)
    coef_mse_rb = mean_squared_error(
        beta_true[mask_rb].flatten(),
        beta_est_robust[mask_rb].flatten()
    )
    tv_rb = calculate_tv(beta_est_robust)

    return {
        "Static_MSE": coef_mse_static,
        "Static_TV": tv_static,
        "TVLS_MSE": coef_mse_ls,
        "TVLS_TV": tv_ls,
        "RMD_MSE": coef_mse_rb,
        "RMD_TV": tv_rb
    }
     

import pandas as pd
from tqdm import trange


#Section 4.2 — Main Simulation (50 replications)
n_reps = 50
T = 200
p = 50
missing_rate = 0.1
contamination = 0.1

all_results = []

for scenario in ['smooth', 'abrupt']:
    print(f"\nRunning scenario: {scenario}")

    for rep in trange(n_reps):
        out = run_one_simulation(
            T=T,
            p=p,
            missing_rate=missing_rate,
            contamination=contamination,
            lam1=0.1,
            lam2=0.3,
            tau_robust=2.0,
            tau_ls=1e6,
            scenario=scenario
        )

        all_results.append({
            "Scenario": scenario,
            "Replication": rep + 1,
            "Static_MSE": out["Static_MSE"],
            "Static_TV": out["Static_TV"],
            "TVLS_MSE": out["TVLS_MSE"],
            "TVLS_TV": out["TVLS_TV"],
            "RMD_MSE": out["RMD_MSE"],
            "RMD_TV": out["RMD_TV"]
        })

# Put results into a DataFrame
results_df = pd.DataFrame(all_results)

display(results_df.head())

print("\nSummary statistics by scenario:")
summary_df = results_df.groupby("Scenario").agg(["mean", "std"])
print(summary_df)

In [ ]:
# ============================================================
# Real Data Part

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zipfile
import io
import requests

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error



# Helper: total variation for each coefficient

def total_variation_per_coefficient(beta):
    """
    Column-wise total variation for each coefficient path.
    Returns a vector of length p.
    """
    return np.sum(np.abs(np.diff(beta, axis=0)), axis=0)


# ============================================================
# Part A. Bike Sharing Dataset
# ============================================================

# 1. Load Bike Sharing data

url = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))
df_bike = pd.read_csv(z.open("hour.csv"))

print("Bike raw data shape:", df_bike.shape)
print(df_bike.head())



# 2. Downsample to daily observations

df_bike_daily = df_bike.iloc[::24, :].copy()

y_bike = df_bike_daily["cnt"].values.astype(float)
X_bike = df_bike_daily[["temp", "atemp", "hum", "windspeed"]].values.astype(float)

T_bike = len(y_bike)
p_bike = X_bike.shape[1]

print("\nBike dataset:")
print("T =", T_bike, "   p =", p_bike)

# Standardize predictors
scaler_bike = StandardScaler()
X_bike_scaled = scaler_bike.fit_transform(X_bike)



# 3. Fit RMD-TVReg on Bike data and plot coefficient paths

beta_bike_rmd = rmd_tvreg_fit(
    X=X_bike_scaled,
    y=y_bike,
    lam1=0.05,
    lam2=0.05,
    tau=2.0,
    max_iter=60,
    cd_iter=20,
    is_static=False
)

labels_bike = ["temp", "atemp", "hum", "windspeed"]
line_styles_bike = ["solid", "dashed", "solid", "dashdot"]
colors_bike = ["blue", "orange", "green", "red"]

plt.figure(figsize=(10, 5))

for j in range(p_bike):
    plt.plot(
        beta_bike_rmd[:, j],
        label=labels_bike[j],
        linewidth=2.5,
        linestyle=line_styles_bike[j],
        color=colors_bike[j]
    )

plt.xlabel("Time", fontsize=18)
plt.ylabel("Coefficient Value", fontsize=18)
plt.title("Time-varying coefficients on Bike dataset", fontsize=18)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.legend(fontsize=16)
plt.tight_layout()
plt.savefig("bike_coefficients.pdf", format="pdf", bbox_inches="tight")
plt.show()



# 4. Compare total variation of TV-LS and RMD-TVReg

beta_bike_tvls = rmd_tvreg_fit(
    X=X_bike_scaled,
    y=y_bike,
    lam1=0.05,
    lam2=0.05,
    tau=1e6,       
    max_iter=60,
    cd_iter=20,
    is_static=False
)

beta_bike_rmd = rmd_tvreg_fit(
    X=X_bike_scaled,
    y=y_bike,
    lam1=0.05,
    lam2=0.05,
    tau=2.0,
    max_iter=60,
    cd_iter=20,
    is_static=False
)

tv_bike_tvls = total_variation_per_coefficient(beta_bike_tvls)
tv_bike_rmd  = total_variation_per_coefficient(beta_bike_rmd)

print("\nBike dataset: Total Variation by coefficient")
for j, name in enumerate(labels_bike):
    print(f"{name:10s}: TV-LS = {tv_bike_tvls[j]:.4f}   RMD-TVReg = {tv_bike_rmd[j]:.4f}")



# 5. Rolling one-step-ahead forecast on Bike data

T_bike = len(y_bike)
train_size_bike = int(0.8 * T_bike)

errors_bike_tvls = []
errors_bike_rmd = []

for t in range(train_size_bike, T_bike):
    # Train on observations up to time t-1
    X_train = X_bike_scaled[:t]
    y_train = y_bike[:t]

    # Fit TV-LS
    beta_tvls_t = rmd_tvreg_fit(
        X=X_train,
        y=y_train,
        lam1=0.05,
        lam2=0.05,
        tau=1e6,
        max_iter=40,
        cd_iter=10,
        is_static=False
    )

    # Fit RMD-TVReg
    beta_rmd_t = rmd_tvreg_fit(
        X=X_train,
        y=y_train,
        lam1=0.05,
        lam2=0.05,
        tau=2.0,
        max_iter=40,
        cd_iter=10,
        is_static=False
    )

    # One-step-ahead prediction for time t
    x_next = X_bike_scaled[t]
    y_true = y_bike[t]

    y_pred_tvls = np.dot(x_next, beta_tvls_t[-1])
    y_pred_rmd  = np.dot(x_next, beta_rmd_t[-1])

    errors_bike_tvls.append(y_true - y_pred_tvls)
    errors_bike_rmd.append(y_true - y_pred_rmd)

rmse_bike_tvls = np.sqrt(mean_squared_error(np.zeros_like(errors_bike_tvls), errors_bike_tvls))
rmse_bike_rmd  = np.sqrt(mean_squared_error(np.zeros_like(errors_bike_rmd), errors_bike_rmd))

mae_bike_tvls = mean_absolute_error(np.zeros_like(errors_bike_tvls), errors_bike_tvls)
mae_bike_rmd  = mean_absolute_error(np.zeros_like(errors_bike_rmd), errors_bike_rmd)

print("\nBike dataset: Rolling one-step-ahead forecast errors")
print(f"TV-LS:     RMSE = {rmse_bike_tvls:.4f},  MAE = {mae_bike_tvls:.4f}")
print(f"RMD-TVReg: RMSE = {rmse_bike_rmd:.4f},  MAE = {mae_bike_rmd:.4f}")


# ============================================================
# Part B. Air Quality Dataset
# ============================================================


# 1. Load and clean Air Quality data

df_raw = pd.read_csv("AirQualityUCI.csv", sep=",", encoding="latin-1")

# Drop extra unnamed columns if present
for col in ["Unnamed: 15", "Unnamed: 16"]:
    if col in df_raw.columns:
        df_raw = df_raw.drop(columns=[col])

# Fix BOM in first column name if present
if "ï»¿Date" in df_raw.columns:
    df_raw = df_raw.rename(columns={"ï»¿Date": "Date"})

print("\nAirQuality raw data shape:", df_raw.shape)
print(df_raw.head())
print(df_raw.columns)

# Replace -200 with NaN
df_air = df_raw.replace(-200, np.nan)

# Downsample to daily observations
df_air_daily = df_air.iloc[::24, :].copy()

# Response and predictors
y_col = "NO2(GT)"
X_cols = [
    "PT08.S1(CO)", "PT08.S2(NMHC)", "PT08.S3(NOx)",
    "PT08.S4(NO2)", "PT08.S5(O3)", "T", "RH", "AH"
]

# Keep complete rows
df_air_sub = df_air_daily[[y_col] + X_cols].dropna().copy()

y_air = df_air_sub[y_col].values.astype(float)
X_air = df_air_sub[X_cols].values.astype(float)

T_air = len(y_air)
p_air = X_air.shape[1]

print("\nAir dataset:")
print("T =", T_air, "   p =", p_air)

# Standardize predictors
scaler_air = StandardScaler()
X_air_scaled = scaler_air.fit_transform(X_air)



# 2. Fit TV-LS and RMD-TVReg on Air data

beta_air_tvls = rmd_tvreg_fit(
    X=X_air_scaled,
    y=y_air,
    lam1=0.05,
    lam2=0.05,
    tau=1e6,
    max_iter=40,
    cd_iter=15,
    is_static=False
)

beta_air_rmd = rmd_tvreg_fit(
    X=X_air_scaled,
    y=y_air,
    lam1=0.05,
    lam2=0.05,
    tau=2.0,
    max_iter=40,
    cd_iter=15,
    is_static=False
)

print("\nAir coefficient path shapes:")
print("beta_air_tvls:", beta_air_tvls.shape)
print("beta_air_rmd :", beta_air_rmd.shape)



# 3. Plot selected coefficient paths on Air data

selected_cols = ["PT08.S4(NO2)", "PT08.S5(O3)", "T", "RH"]
selected_idx = [X_cols.index(c) for c in selected_cols]

labels_air = ["NO2 sensor", "O3 sensor", "Temperature (T)", "Relative Humidity (RH)"]
line_styles_air = ["solid", "dashed", "dotted", "dashdot"]

plt.figure(figsize=(10, 5))

for j, idx in enumerate(selected_idx):
    plt.plot(
        beta_air_rmd[:, idx],
        label=labels_air[j],
        linewidth=2.5,
        linestyle=line_styles_air[j]
    )

plt.xlabel("Time", fontsize=18)
plt.ylabel("Coefficient value", fontsize=18)
plt.title("Time-varying coefficients on Air Quality dataset (RMD-TVReg)", fontsize=18)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.legend(fontsize=14)
plt.tight_layout()
plt.savefig("air_coefficients.pdf", format="pdf", bbox_inches="tight")
plt.show()



# 4. Compare total variation of TV-LS and RMD-TVReg on Air data

tv_air_tvls = total_variation_per_coefficient(beta_air_tvls)
tv_air_rmd  = total_variation_per_coefficient(beta_air_rmd)

print("\nAir dataset: Total variation by coefficient")
for name, v_ls, v_rmd in zip(X_cols, tv_air_tvls, tv_air_rmd):
    print(f"{name:15s}  TV-LS = {v_ls:.4f}   RMD-TVReg = {v_rmd:.4f}")
